# 基于 BP 神经网络的二维真实数据分类实验（学生练习版）

本练习使用真实的 **Iris 鸢尾花数据集**，选择两个真实测量特征构成二维输入，并使用一个包含 **输入层、隐藏层和输出层** 的前馈神经网络完成二分类。

本练习已经保留：

1. Iris 数据集读取。
2. 数据可视化。
3. 训练集与测试集划分。
4. 数据标准化。
5. 参数初始化。
6. 训练框架。
7. 分类边界可视化。
8. 误差变化分析。

需要学生补充的关键内容：

1. `sigmoid()` 激活函数。
2. `forward_propagation()` 前向传播。
3. `compute_loss()` 二元交叉熵损失。
4. `backward_propagation()` 反向传播梯度。
5. `update_parameters()` 梯度下降参数更新。

完成 TODO 后，重新从头运行 Notebook，即可训练 BP 神经网络并观察分类效果。

## 1. BP 神经网络相关背景知识

### 1.1 前馈神经网络

前馈神经网络 Feedforward Neural Network 是最基本的神经网络结构之一。数据从输入层进入，依次经过隐藏层，最后到达输出层。

本实验使用一个三层网络：

```text
输入层  →  隐藏层  →  输出层
 2维        H个神经元    1个神经元
```

对于二维分类问题，每个样本可以表示为：

$$
x = [x_1, x_2]
$$

网络输出一个概率值：

$$
\hat{y} \in (0, 1)
$$

当 $\hat{y} \ge 0.5$ 时，认为样本属于类别 1；否则属于类别 0。

### 1.2 神经元与激活函数

一个神经元通常先进行线性变换，再通过非线性激活函数：

$$
z = xW + b
$$

$$
a = f(z)
$$

其中：

- `x`：输入特征。
- `W`：权重。
- `b`：偏置。
- `z`：线性输出。
- `a`：激活后的输出。
- `f`：激活函数。

本实验隐藏层使用 `tanh` 激活函数：

$$
\tanh(z)=\frac{e^z-e^{-z}}{e^z+e^{-z}}
$$

输出层使用 `sigmoid` 激活函数：

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

### 1.3 前向传播

前向传播的作用是：根据当前参数计算模型输出。

本实验网络的前向传播过程为：

$$
Z_1 = XW_1 + b_1
$$

$$
A_1 = \tanh(Z_1)
$$

$$
Z_2 = A_1W_2 + b_2
$$

$$
\hat{Y}=A_2=\sigma(Z_2)
$$

其中：

- `X`：输入数据，形状为 `(样本数, 2)`。
- `W1`：输入层到隐藏层的权重。
- `b1`：隐藏层偏置。
- `W2`：隐藏层到输出层的权重。
- `b2`：输出层偏置。
- `A2`：网络输出的类别 1 概率。

### 1.4 损失函数

对于二分类任务，常用二元交叉熵损失 Binary Cross Entropy：

$$
L = -\frac{1}{m}\sum_{i=1}^{m}
\left[
y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)
\right]
$$

其中：

- `m`：样本数量。
- `y_i`：真实标签，取值为 0 或 1。
- `ŷ_i`：模型预测为类别 1 的概率。

损失越小，说明预测结果与真实标签越接近。

### 1.5 反向传播

反向传播 Back Propagation, BP 的作用是：根据损失函数对各层参数求梯度，然后使用梯度下降更新参数。

本实验中需要计算：

$$
\frac{\partial L}{\partial W_2},\quad
\frac{\partial L}{\partial b_2},\quad
\frac{\partial L}{\partial W_1},\quad
\frac{\partial L}{\partial b_1}
$$

对于 sigmoid 输出层和二元交叉熵损失，有一个简洁结果：

$$
dZ_2 = A_2 - Y
$$

然后逐层向前传播梯度：

$$
dW_2 = \frac{1}{m}A_1^T dZ_2
$$

$$
db_2 = \frac{1}{m}\sum dZ_2
$$

$$
dA_1 = dZ_2 W_2^T
$$

因为：

$$
\frac{d}{dz}\tanh(z)=1-\tanh^2(z)
$$

所以：

$$
dZ_1 = dA_1 \cdot (1-A_1^2)
$$

$$
dW_1 = \frac{1}{m}X^T dZ_1
$$

$$
db_1 = \frac{1}{m}\sum dZ_1
$$

## 2. 学生任务：完成前向计算与反向传播

在运行后续训练程序之前，请先按照下面的步骤补全代码中的 TODO。

### 2.1 前向传播应该先做什么、再做什么？

前向传播的目标是：从输入数据 `X` 出发，计算网络输出 `A2`，并保存反向传播需要的中间变量。

请按顺序完成：

```text
输入：X, W1, b1, W2, b2

步骤 1：计算隐藏层线性输出
        Z1 = X @ W1 + b1

步骤 2：计算隐藏层激活值
        A1 = tanh(Z1)

步骤 3：计算输出层线性输出
        Z2 = A1 @ W2 + b2

步骤 4：计算输出层预测概率
        A2 = sigmoid(Z2)

步骤 5：保存 X, Z1, A1, Z2, A2 到 cache

输出：A2, cache
```

### 2.2 损失函数应该怎么计算？

二分类交叉熵损失：

```text
输入：真实标签 y_true，预测概率 y_pred

步骤 1：为了避免 log(0)，先把 y_pred 限制在 [eps, 1-eps]
步骤 2：计算每个样本的交叉熵
步骤 3：对所有样本取平均

输出：loss
```

公式：

$$
L = -\frac{1}{m}\sum_{i=1}^{m}
\left[
y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)
\right]
$$

### 2.3 反向传播应该先做什么、再做什么？

反向传播的目标是：根据损失函数，计算每个参数的梯度。

请按顺序完成：

```text
输入：y_true, params, cache

步骤 1：从 cache 中取出 X, A1, A2
        从 params 中取出 W2
        记样本数 m = X.shape[0]

步骤 2：计算输出层误差
        dZ2 = A2 - y_true

步骤 3：计算输出层参数梯度
        dW2 = A1.T @ dZ2 / m
        db2 = sum(dZ2) / m

步骤 4：把误差传回隐藏层
        dA1 = dZ2 @ W2.T

步骤 5：结合 tanh 的导数计算隐藏层误差
        dZ1 = dA1 * (1 - A1 ** 2)

步骤 6：计算输入层到隐藏层的参数梯度
        dW1 = X.T @ dZ1 / m
        db1 = sum(dZ1) / m

步骤 7：把 dW1, db1, dW2, db2 保存到 grads

输出：grads
```

### 2.4 参数更新应该先做什么、再做什么？

梯度下降更新规则：

```text
W1 = W1 - learning_rate * dW1
b1 = b1 - learning_rate * db1
W2 = W2 - learning_rate * dW2
b2 = b2 - learning_rate * db2
```

补全所有 TODO 后，再运行训练、误差曲线和分类边界可视化代码。

## 3. 实验准备

下面导入基础库。本实验只使用 `numpy` 进行数值计算，使用 `matplotlib` 进行可视化。

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4. 读取真实二维分类数据集

本实验使用 **Iris 鸢尾花数据集**。该数据集由 R. A. Fisher 提出，是机器学习中非常经典的真实分类数据集。

Iris 原始数据包含 150 个样本、3 个鸢尾花类别和 4 个测量特征：

| 特征 | 含义 |
|---|---|
| sepal length | 萼片长度 |
| sepal width | 萼片宽度 |
| petal length | 花瓣长度 |
| petal width | 花瓣宽度 |

为了保持二维分类实验的直观性，本实验只选择两个真实特征：

$$
x = [\text{petal length}, \text{petal width}]
$$

同时只选取两个类别进行二分类：

- `Iris-setosa`：记为类别 0
- `Iris-versicolor`：记为类别 1

这样就得到一个真实来源的二维二分类数据集，而不是人工生成的数据。

In [ ]:
IRIS_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
IRIS_LOCAL_PATH = Path("iris_data") / "iris.data"


def download_iris_if_needed(url=IRIS_URL, local_path=IRIS_LOCAL_PATH):
    """
    下载 Iris 数据集。如果本地已经存在，则直接使用缓存文件。

    参数：
        url: UCI Iris 数据集下载地址
        local_path: 本地保存路径
    """
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists():
        print(f"使用本地缓存数据：{local_path}")
        return

    print(f"正在下载 Iris 数据集：{url}")
    urllib.request.urlretrieve(url, local_path)
    print(f"数据已保存到：{local_path}")


def load_iris_binary_2d(local_path=IRIS_LOCAL_PATH, random_state=42):
    """
    读取 Iris 数据集，并转换为二维二分类任务。

    参数：
        local_path: Iris CSV 数据文件路径
        random_state: 随机种子

    返回：
        X: 二维真实特征，形状为 (100, 2)
        y: 二分类标签，形状为 (100, 1)
        feature_names: 使用的特征名称
        class_names: 类别名称
    """
    download_iris_if_needed(local_path=local_path)

    class_to_label = {
        "Iris-setosa": 0,
        "Iris-versicolor": 1,
    }

    X_list = []
    y_list = []

    for line in local_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue

        parts = line.split(",")
        if len(parts) != 5:
            continue

        sepal_length, sepal_width, petal_length, petal_width, species = parts
        if species not in class_to_label:
            continue

        # 只选择两个真实测量特征：花瓣长度、花瓣宽度
        X_list.append([float(petal_length), float(petal_width)])
        y_list.append([class_to_label[species]])

    X = np.array(X_list, dtype=np.float64)
    y = np.array(y_list, dtype=np.int64)

    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(X))
    X = X[indices]
    y = y[indices]

    feature_names = ["花瓣长度 petal length", "花瓣宽度 petal width"]
    class_names = ["Iris-setosa", "Iris-versicolor"]
    return X, y, feature_names, class_names


X, y, feature_names, class_names = load_iris_binary_2d(random_state=RANDOM_STATE)

print("数据集名称：Iris 鸢尾花数据集")
print("使用类别：", class_names)
print("使用特征：", feature_names)
print("X 形状：", X.shape)
print("y 形状：", y.shape)
print("前 5 个样本：")
print(X[:5])
print("前 5 个标签：")
print(y[:5].ravel())

## 5. 数据集可视化

下面展示真实 Iris 数据集中两类鸢尾花在两个真实特征上的分布。

In [ ]:
def plot_dataset(X, y, title="Iris 二维真实分类数据集", xlabel="x1", ylabel="x2"):
    """
    绘制二维数据集散点图。

    参数：
        X: 二维特征，形状为 (N, 2)
        y: 标签，形状为 (N, 1)
        title: 图标题
        xlabel: 横轴名称
        ylabel: 纵轴名称
    """
    y_flat = y.ravel()
    plt.figure(figsize=(6, 5))
    plt.scatter(X[y_flat == 0, 0], X[y_flat == 0, 1], c="#2A6FBB", label=class_names[0], edgecolors="k", s=45)
    plt.scatter(X[y_flat == 1, 0], X[y_flat == 1, 1], c="#E76F51", label=class_names[1], edgecolors="k", s=45)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


plot_dataset(
    X,
    y,
    title="Iris 数据集：花瓣长度与花瓣宽度",
    xlabel=feature_names[0],
    ylabel=feature_names[1],
)

## 6. 划分训练集和测试集

手动打乱数据，并划分为训练集和测试集。

In [ ]:
def train_test_split_manual(X, y, test_ratio=0.25, random_state=42):
    """
    手动划分训练集和测试集。

    参数：
        X: 特征矩阵
        y: 标签
        test_ratio: 测试集比例
        random_state: 随机种子

    返回：
        X_train, y_train, X_test, y_test
    """
    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(X))
    test_size = int(len(X) * test_ratio)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    return X[train_indices], y[train_indices], X[test_indices], y[test_indices]


X_train, y_train, X_test, y_test = train_test_split_manual(X, y, test_ratio=0.25, random_state=RANDOM_STATE)

print("训练集：", X_train.shape, y_train.shape)
print("测试集：", X_test.shape, y_test.shape)

## 7. 数据标准化

神经网络训练通常对数据尺度比较敏感。这里使用训练集的均值和标准差对训练集、测试集进行标准化。

注意：标准化参数只能由训练集计算，再应用到训练集和测试集，不能用测试集信息参与训练。

In [ ]:
def fit_standardizer(X):
    """
    根据训练集计算均值和标准差。

    参数：
        X: 训练特征

    返回：
        mean: 均值
        std: 标准差
    """
    mean = X.mean(axis=0, keepdims=True)
    std = X.std(axis=0, keepdims=True)
    std[std < 1e-8] = 1.0
    return mean, std


def transform_standardize(X, mean, std):
    """
    对数据进行标准化。

    参数：
        X: 待标准化数据
        mean: 训练集均值
        std: 训练集标准差

    返回：
        标准化后的数据
    """
    return (X - mean) / std


mean, std = fit_standardizer(X_train)
X_train_std = transform_standardize(X_train, mean, std)
X_test_std = transform_standardize(X_test, mean, std)

plot_dataset(
    X_train_std,
    y_train,
    title="标准化后的 Iris 训练集",
    xlabel="标准化花瓣长度",
    ylabel="标准化花瓣宽度",
)

## 8. 实现 BP 神经网络：请补全 TODO

下面代码已经给出函数结构、参数说明和返回值说明。请在 `TODO` 位置补全 BP 神经网络的核心算法。

网络结构：

```text
输入层：2 个节点
隐藏层：hidden_size 个节点，tanh 激活
输出层：1 个节点，sigmoid 激活
```

建议补全顺序：

1. 先完成 `sigmoid()`。
2. 再完成 `forward_propagation()`。
3. 再完成 `compute_loss()`。
4. 再完成 `backward_propagation()`。
5. 最后完成 `update_parameters()`。

In [ ]:
def sigmoid(z):
    """
    Sigmoid 激活函数。

    参数：
        z: 输入数组

    返回：
        sigmoid(z)
    """
    # TODO 1：
    # 为了避免 exp 溢出，可以先使用 np.clip 将 z 限制在 [-50, 50]
    # 然后根据公式 sigmoid(z) = 1 / (1 + exp(-z)) 返回结果
    raise NotImplementedError("请补全 sigmoid 函数")


def initialize_parameters(input_size, hidden_size, output_size, random_state=42):
    """
    初始化网络参数。

    参数：
        input_size: 输入层节点数
        hidden_size: 隐藏层节点数
        output_size: 输出层节点数
        random_state: 随机种子

    返回：
        params: 参数字典，包含 W1, b1, W2, b2
    """
    rng = np.random.default_rng(random_state)
    params = {
        "W1": rng.normal(0, 0.5, size=(input_size, hidden_size)),
        "b1": np.zeros((1, hidden_size)),
        "W2": rng.normal(0, 0.5, size=(hidden_size, output_size)),
        "b2": np.zeros((1, output_size)),
    }
    return params


def forward_propagation(X, params):
    """
    前向传播。

    参数：
        X: 输入特征，形状为 (N, 2)
        params: 网络参数字典

    返回：
        A2: 输出层预测概率，形状为 (N, 1)
        cache: 前向传播中间变量，用于反向传播
    """
    W1, b1 = params["W1"], params["b1"]
    W2, b2 = params["W2"], params["b2"]

    # TODO 2：
    # 按顺序完成以下计算：
    # Z1 = X @ W1 + b1
    # A1 = np.tanh(Z1)
    # Z2 = A1 @ W2 + b2
    # A2 = sigmoid(Z2)

    # TODO 3：
    # 将 X, Z1, A1, Z2, A2 保存到 cache 字典中
    # cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}

    raise NotImplementedError("请补全 forward_propagation 函数")


def compute_loss(y_true, y_pred):
    """
    计算二元交叉熵损失。

    参数：
        y_true: 真实标签，形状为 (N, 1)
        y_pred: 预测概率，形状为 (N, 1)

    返回：
        loss: 平均损失
    """
    # TODO 4：
    # 1. 设置 eps = 1e-12
    # 2. 使用 np.clip 避免 log(0)
    # 3. 根据二元交叉熵公式计算平均损失
    raise NotImplementedError("请补全 compute_loss 函数")


def backward_propagation(y_true, params, cache):
    """
    反向传播，计算各参数梯度。

    参数：
        y_true: 真实标签，形状为 (N, 1)
        params: 网络参数字典
        cache: 前向传播保存的中间变量

    返回：
        grads: 梯度字典，包含 dW1, db1, dW2, db2
    """
    # TODO 5：
    # 从 cache 中取出 X, A1, A2
    # 从 params 中取出 W2
    # m = X.shape[0]

    # TODO 6：
    # 输出层梯度：
    # dZ2 = A2 - y_true
    # dW2 = A1.T @ dZ2 / m
    # db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # TODO 7：
    # 隐藏层梯度：
    # dA1 = dZ2 @ W2.T
    # dZ1 = dA1 * (1 - A1 ** 2)
    # dW1 = X.T @ dZ1 / m
    # db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    # TODO 8：
    # 将 dW1, db1, dW2, db2 保存到 grads 字典中并返回

    raise NotImplementedError("请补全 backward_propagation 函数")


def update_parameters(params, grads, learning_rate):
    """
    使用梯度下降更新参数。

    参数：
        params: 参数字典
        grads: 梯度字典
        learning_rate: 学习率

    返回：
        params: 更新后的参数字典
    """
    # TODO 9：
    # 根据梯度下降公式更新 W1, b1, W2, b2：
    # params["W1"] = params["W1"] - learning_rate * grads["dW1"]
    # params["b1"] = params["b1"] - learning_rate * grads["db1"]
    # params["W2"] = params["W2"] - learning_rate * grads["dW2"]
    # params["b2"] = params["b2"] - learning_rate * grads["db2"]
    # return params

    raise NotImplementedError("请补全 update_parameters 函数")

## 9. 训练网络

训练框架已经给出，不作为本练习的主要考查内容。学生只需要先补全上一节中的核心函数，训练代码就可以调用这些函数完成网络训练。

训练过程顺序：

1. 初始化参数。
2. 调用 `forward_propagation()` 执行前向传播，得到预测结果。
3. 调用 `compute_loss()` 计算损失。
4. 调用 `backward_propagation()` 计算梯度。
5. 调用 `update_parameters()` 根据梯度更新参数。
6. 重复多轮训练。
7. 使用训练后的参数进行分类和可视化。

In [ ]:
def predict_proba(X, params):
    """
    预测类别 1 的概率。

    参数：
        X: 输入特征
        params: 网络参数

    返回：
        概率数组
    """
    proba, _ = forward_propagation(X, params)
    return proba


def predict_class(X, params, threshold=0.5):
    """
    预测类别标签。

    参数：
        X: 输入特征
        params: 网络参数
        threshold: 分类阈值

    返回：
        预测标签，形状为 (N, 1)
    """
    proba = predict_proba(X, params)
    return (proba >= threshold).astype(int)


def accuracy_score_manual(y_true, y_pred):
    """
    手动计算分类准确率。

    参数：
        y_true: 真实标签
        y_pred: 预测标签

    返回：
        准确率
    """
    return np.mean(y_true == y_pred)


def train_network(X_train, y_train, X_test, y_test, hidden_size=8, learning_rate=0.08, epochs=3000):
    """
    训练 BP 神经网络。

    参数：
        X_train: 训练特征
        y_train: 训练标签
        X_test: 测试特征
        y_test: 测试标签
        hidden_size: 隐藏层神经元数量
        learning_rate: 学习率
        epochs: 训练轮数

    返回：
        params: 训练后的参数
        history: 训练过程记录
    """
    params = initialize_parameters(
        input_size=X_train.shape[1],
        hidden_size=hidden_size,
        output_size=1,
        random_state=RANDOM_STATE,
    )

    history = {
        "loss": [],
        "train_acc": [],
        "test_acc": [],
    }

    for epoch in range(1, epochs + 1):
        y_pred_proba, cache = forward_propagation(X_train, params)
        loss = compute_loss(y_train, y_pred_proba)
        grads = backward_propagation(y_train, params, cache)
        params = update_parameters(params, grads, learning_rate)

        if epoch % 50 == 0 or epoch == 1:
            train_pred = predict_class(X_train, params)
            test_pred = predict_class(X_test, params)
            train_acc = accuracy_score_manual(y_train, train_pred)
            test_acc = accuracy_score_manual(y_test, test_pred)

            history["loss"].append(loss)
            history["train_acc"].append(train_acc)
            history["test_acc"].append(test_acc)

        if epoch % 500 == 0 or epoch == 1:
            print(f"Epoch {epoch:4d} | loss={loss:.4f} | train_acc={train_acc:.4f} | test_acc={test_acc:.4f}")

    return params, history


params, history = train_network(
    X_train_std,
    y_train,
    X_test_std,
    y_test,
    hidden_size=8,
    learning_rate=0.08,
    epochs=3000,
)

## 10. 误差变化分析

下面绘制训练过程中损失和准确率的变化。

In [ ]:
steps = np.arange(len(history["loss"])) * 50
steps[0] = 1

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(steps, history["loss"], color="#2A6FBB")
plt.xlabel("训练轮数")
plt.ylabel("损失")
plt.title("训练误差变化")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(steps, history["train_acc"], label="训练集准确率", color="#2A6FBB")
plt.plot(steps, history["test_acc"], label="测试集准确率", color="#E76F51")
plt.xlabel("训练轮数")
plt.ylabel("准确率")
plt.title("准确率变化")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. 分类结果可视化

下面绘制神经网络学习到的分类边界。背景颜色表示模型在不同二维位置上的预测类别。

In [ ]:
def plot_decision_boundary(X, y, params, title="BP 神经网络分类边界"):
    """
    绘制二维分类边界。

    参数：
        X: 标准化后的二维特征
        y: 标签
        params: 训练后的网络参数
        title: 图标题
    """
    x_min, x_max = X[:, 0].min() - 0.6, X[:, 0].max() + 0.6
    y_min, y_max = X[:, 1].min() - 0.6, X[:, 1].max() + 0.6

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    pred = predict_class(grid, params).reshape(xx.shape)

    plt.figure(figsize=(7, 6))
    plt.contourf(xx, yy, pred, levels=[-0.5, 0.5, 1.5], colors=["#BBD7F0", "#F4B6A6"], alpha=0.7)

    y_flat = y.ravel()
    plt.scatter(X[y_flat == 0, 0], X[y_flat == 0, 1], c="#2A6FBB", edgecolors="k", label="类别 0", s=35)
    plt.scatter(X[y_flat == 1, 0], X[y_flat == 1, 1], c="#E76F51", edgecolors="k", label="类别 1", s=35)
    plt.xlabel("标准化 x1")
    plt.ylabel("标准化 x2")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


plot_decision_boundary(X_train_std, y_train, params, title="训练集上的分类边界")
plot_decision_boundary(X_test_std, y_test, params, title="测试集上的分类边界")

## 12. 测试集评价

下面输出最终的测试集准确率，并显示部分样本的预测结果。

In [ ]:
test_pred = predict_class(X_test_std, params)
test_acc = accuracy_score_manual(y_test, test_pred)

print(f"测试集准确率：{test_acc:.4f}")

result_preview = np.hstack([X_test[:10], y_test[:10], test_pred[:10]])
print("前 10 个测试样本：[x1, x2, 真实类别, 预测类别]")
print(np.round(result_preview, 3))

## 13. 网络结构与参数影响实验

可以修改以下参数，观察分类边界和误差曲线的变化：

1. `hidden_size`：隐藏层神经元数量。
2. `learning_rate`：学习率。
3. `epochs`：训练轮数。
4. 特征选择：可以尝试把 `load_iris_binary_2d()` 中的特征改为萼片长度、萼片宽度等其他真实特征。

思考：

1. 隐藏层神经元数量太少时，分类边界会发生什么？
2. 学习率太大时，误差曲线会怎样变化？
3. 如果改用 `Iris-versicolor` 和 `Iris-virginica` 两个类别，分类难度会怎样变化？
4. 使用花瓣特征和萼片特征，哪一组特征更容易区分这两类鸢尾花？

In [ ]:
# 可以在这里快速尝试不同隐藏层规模
hidden_sizes = [2, 4, 8, 16]
experiment_results = []

for h in hidden_sizes:
    temp_params, temp_history = train_network(
        X_train_std,
        y_train,
        X_test_std,
        y_test,
        hidden_size=h,
        learning_rate=0.08,
        epochs=1500,
    )
    temp_pred = predict_class(X_test_std, temp_params)
    temp_acc = accuracy_score_manual(y_test, temp_pred)
    experiment_results.append((h, temp_acc))

print("隐藏层规模对测试准确率的影响：")
for h, acc in experiment_results:
    print(f"hidden_size={h:2d}, test_acc={acc:.4f}")

## 14. 实验小结

本实验使用真实的 Iris 鸢尾花数据集完成了一个基于 BP 神经网络的二维数据分类任务。

通过本实验可以看到：

1. 真实数据集可以通过选择两个测量特征转化为二维分类问题。
2. 前馈神经网络通过前向传播计算预测结果。
3. BP 反向传播通过链式法则计算各层参数梯度。
4. 梯度下降根据梯度不断更新参数，使损失逐渐下降。
5. 隐藏层和非线性激活函数使网络能够学习非线性分类边界。
6. 误差曲线和分类边界可视化能够帮助分析模型训练效果。

本实验中的网络规模较小，但已经包含神经网络训练的核心流程：前向传播、损失计算、反向传播和参数更新。